In [ ]:
%matplotlib inline

# Qubit tapering

The performance of variational quantum algorithms is considerably
limited by the number of qubits required to represent wave functions. In
the context of quantum chemistry, this limitation hinders the treatment
of large molecules with algorithms such as the [variational quantum
eigensolver (VQE)](https://pennylane.ai/qml/demos/tutorial_vqe). Several
approaches have been developed to reduce the qubit requirements for
quantum chemistry calculations. In this tutorial, we demonstrate the
symmetry-based qubit tapering approach which allows reducing the number
of qubits required to perform molecular quantum simulations based on the
$\mathbb{Z}_2$ symmetries present in molecular Hamiltonians .

A molecular Hamiltonian in the qubit basis can be expressed as a linear
combination of Pauli words as

$$H = \sum_{i=1}^r h_i P_i,$$

where $h_i$ is a real coefficient and $P_i$ is a tensor product of Pauli
and identity operators acting on $M$ qubits

$$P_i \in \pm \left \{ I, X, Y, Z \right \} ^ {\bigotimes M}.$$

The main idea in the symmetry-based qubit tapering approach is to find a
unitary operator $U$ that transforms $H$ to a new Hamiltonian $H'$ which
has the same eigenvalues as $H$

$$H' = U^{\dagger} H U = \sum_{i=1}^r c_i \mu_i,$$

such that each $\mu_i$ term in the new Hamiltonian always acts
trivially, e.g., with an identity or a Pauli operator, on a set of
qubits. This allows tapering-off those qubits from the Hamiltonian.

For instance, consider the following Hamiltonian

$$H' = Z_0 X_1 - X_1 + Y_0 X_1,$$

where all terms in the Hamiltonian act on the second qubit with the $X$
operator. It is straightforward to show that each term in the
Hamiltonian commutes with $I_0 X_1$ and the ground-state eigenvector of
$H'$ is also an eigenvector of $I_0 X_1$ with eigenvalues $\pm 1.$ We
can also rewrite the Hamiltonian as

$$H' = (Z_0 I_1 - I_0 I_1 + Y_0 I_1) I_0 X_1,$$

which gives us

$$H'|\psi \rangle = \pm1 (Z_0 I_1 - I_0 I_1 + Y_0 I_1)|\psi \rangle,$$

where $|\psi \rangle$ is an eigenvector of $H'.$ This means that the
Hamiltonian $H$ can be simplified as

$$H_{tapered} = \pm1 (Z_0 - I_0 + Y_0).$$

The tapered Hamiltonian $H_{tapered}$ has the eigenvalues

$$[-2.41421, 0.41421],$$

and

$$[2.41421, -0.41421],$$

depending on the value of the $\pm 1$ prefactor. The eigenvalues of the
original Hamiltonian $H$ are

$$[2.41421, -2.41421,  0.41421, -0.41421],$$

which are thus reproduced by the tapered Hamiltonian.

More generally, we can construct the unitary $U$ such that each $\mu_i$
term acts with a Pauli-X operator on a set of qubits
$\left \{ j \right \}, j \in \left \{ l, ..., k \right \}$ where $j$ is
the qubit label. This guarantees that each term of the transformed
Hamiltonian commutes with each of the Pauli-X operators applied to the
$j$-th qubit:

$$[H', X^j] = 0,$$

and the eigenvectors of the transformed Hamiltonian $H'$ are also
eigenvectors of each of the $X^{j}$ operators. Then we can factor out
all of the $X^{j}$ operators from the transformed Hamiltonian and
replace them with their eigenvalues $\pm 1.$ This gives us a set of
tapered Hamiltonians depending on which eigenvalue $\pm 1$ we chose for
each of the $X^{j}$ operators. For instance, in the case of two tapered
qubits, we have four eigenvalue sectors: $[+1, +1]$, $[-1, +1]$,
$[+1, -1],$ $[-1, -1].$ In these tapered Hamiltonians, the set of
$\left \{ j \right \}, j \in \left \{ l, ..., k \right \}$ qubits are
eliminated. For tapered molecular Hamiltonians, it is possible to
determine the optimal sector of the eigenvalues that corresponds to the
ground state. This is explained in more detail in the following
sections.

The unitary operator $U$ can be constructed as a
[Clifford](https://en.wikipedia.org/wiki/Clifford_gates) operator

$$U = \Pi_j \left [\frac{1}{\sqrt{2}} \left (X^{q(j)} + \tau_j \right) \right],$$

where $\tau$ denotes the generators of the symmetry group of $H$ and
$X^{q}$ operators act on those qubits that will be ultimately tapered
off from the Hamiltonian. The symmetry group of the Hamiltonian is
defined as an Abelian group of Pauli words that commute with each term
in the Hamiltonian (excluding $−I$). The
[generators](https://en.wikipedia.org/wiki/Generating_set_of_a_group) of
the symmetry group are those elements of the group that can be combined,
along with their inverses, to create any other member of the group.

Let\'s use the qubit tapering method and obtain the ground state energy
of the [Helium hydride
cation](https://en.wikipedia.org/wiki/Helium_hydride_ion)
$\textrm{HeH}^+.$

## Tapering the molecular Hamiltonian

In PennyLane, a [molecular
Hamiltonian](https://pennylane.ai/qml/demos/tutorial_quantum_chemistry)
can be created by specifying the atomic symbols and coordinates.


In [ ]:
import pennylane as qp
from jax import numpy as jnp
import jax

jax.config.update("jax_enable_x64", True)
symbols = ["He", "H"]
geometry = jnp.array([[0.00000000, 0.00000000, -0.87818361],
                     [0.00000000, 0.00000000,  0.87818362]])

molecule = qp.qchem.Molecule(symbols, geometry, charge=1)
H, qubits = qp.qchem.molecular_hamiltonian(molecule)
H

This Hamiltonian contains 27 terms where each term acts on up to four
qubits.

We can now obtain the symmetry generators and the $X^{j}$ operators that
are used to construct the unitary $U$ operator that transforms the
$\textrm{HeH}^+$ Hamiltonian. In PennyLane, these are constructed by
using the
[symmetry_generators](https://docs.pennylane.ai/en/stable/code/api/pennylane.symmetry_generators.html#)
and
[paulix_ops](https://docs.pennylane.ai/en/stable/code/api/pennylane.paulix_ops.html#)
functions.


In [ ]:
generators = qp.symmetry_generators(H)
paulixops = qp.paulix_ops(generators, qubits)

for idx, generator in enumerate(generators):
    print(f"generator {idx+1}: {generator}, paulix_op: {paulixops[idx]}")

Once the operator $U$ is applied, each of the Hamiltonian terms will act
on the qubits $q_2, q_3$ either with the identity or with a Pauli-X
operator. For each of these qubits, we can simply replace the Pauli-X
operator with one of its eigenvalues $+1$ or $-1.$ This results in a
total number of $2^k$ Hamiltonians, where $k$ is the number of
tapered-off qubits and each Hamiltonian corresponds to one eigenvalue
sector. The optimal sector corresponding to the ground-state energy of
the molecule can be obtained by using the
[optimal_sector](https://docs.pennylane.ai/en/stable/code/api/pennylane.qchem.optimal_sector.html#)
function.


In [ ]:
n_electrons = 2
paulix_sector = qp.qchem.optimal_sector(H, generators, n_electrons)
print(paulix_sector)

The optimal eigenvalues are $-1, -1$ for qubits $q_2, q_3,$
respectively. We can now build the tapered Hamiltonian with the
[taper](https://docs.pennylane.ai/en/stable/code/api/pennylane.taper.html#)
function which constructs the operator $U,$ applies it to the
Hamiltonian and finally tapers off the qubits $q_2, q_3$ by replacing
the Pauli-X operators acting on those qubits with the optimal
eigenvalues.


In [ ]:
H_tapered = qp.taper(H, generators, paulixops, paulix_sector)
H_tapered_coeffs, H_tapered_ops = H_tapered.terms()
H_tapered = qp.Hamiltonian(jnp.real(jnp.array(H_tapered_coeffs)), H_tapered_ops)
print(H_tapered)

The new Hamiltonian has only 9 non-zero terms acting on only 2 qubits!
We can verify that the original and the tapered Hamiltonian both give
the correct ground state energy of the $\textrm{HeH}^+$ cation, which is
$-2.862595242378$ Ha computed with the full configuration interaction
(FCI) method. In PennyLane, it\'s possible to build a sparse matrix
representation of Hamiltonians. This allows us to directly diagonalize
them to obtain exact values of the ground-state energies.


In [ ]:
H_sparse = qp.SparseHamiltonian(H.sparse_matrix(), wires=H.wires)
H_tapered_sparse = qp.SparseHamiltonian(H_tapered.sparse_matrix(), wires=H_tapered.wires)

print("Eigenvalues of H:\n", qp.eigvals(H_sparse, k=16))
print("\nEigenvalues of H_tapered:\n", qp.eigvals(H_tapered_sparse, k=4))

Note that a second-quantized Hamiltonian is independent of the number of
electrons and its eigenspectrum contains the energies of the neutral and
charged molecules. Therefore, the smallest eigenvalue returned by
[eigvals](https://docs.pennylane.ai/en/stable/code/api/pennylane.eigvals.html#)
for a molecular Hamiltonian might correspond to the neutral or charged
molecule. While in the case of $\textrm{HeH}^+,$ qubit tapering allows
specifying the optimal sector of the eigenvectors corresponding only to
the correct number of electrons, it is generally guaranteed that the
optimal sector covers all eigenvectors with the correct number of
electrons, but may contain additional eigenvectors of different charge.
Therefore, the ground-state energy of the $\textrm{HeH}^+$ cation is the
smallest eigenvalue of the tapered Hamiltonian.

# Tapering the reference state

The ground state Hartree-Fock energy of $\textrm{HeH}^+$ can be computed
by directly applying the Hamiltonians to the Hartree-Fock state. For the
tapered Hamiltonian, this requires transforming the Hartree-Fock state
with the same symmetries obtained for the original Hamiltonian. This
reduces the number of qubits in the Hartree-Fock state to match that of
the tapered Hamiltonian. It can be done with the
[taper_hf](https://docs.pennylane.ai/en/stable/code/api/pennylane.qchem.taper_hf.html#)
function.


In [ ]:
state_tapered = qp.qchem.taper_hf(generators, paulixops, paulix_sector,
                                   num_electrons=n_electrons, num_wires=len(H.wires))
print(state_tapered)

Recall that the original Hartree-Fock state for the $\textrm{HeH}^+$
cation is $[1 1 0 0].$ We can now generate the qubit representation of
these states and compute the Hartree-Fock energies for each Hamiltonian.


In [ ]:
dev = qp.device("default.qubit", wires=H.wires)
@qp.qnode(dev, interface="jax")
def circuit():
    qp.BasisState(jnp.array([1, 1, 0, 0]), wires=H.wires)
    return qp.state()

qubit_state = circuit()
HF_energy = qubit_state.T @ H.sparse_matrix().toarray() @ qubit_state
print(f"HF energy: {jnp.real(HF_energy):.8f} Ha")

dev = qp.device("lightning.qubit", wires=H_tapered.wires)
@qp.qnode(dev, interface="jax")
def circuit():
    qp.BasisState(jnp.array([1, 1]), wires=H_tapered.wires)
    return qp.state()

qubit_state = circuit()
HF_energy = qubit_state.T @ H_tapered.sparse_matrix().toarray() @ qubit_state
print(f"HF energy (tapered): {jnp.real(HF_energy):.8f} Ha")

These values are identical to the reference Hartree-Fock energy
$-2.8543686493$ Ha.

# VQE simulation

Finally, we can use the tapered Hamiltonian and the tapered reference
state to perform a VQE simulation and compute the ground-state energy of
the $\textrm{HeH}^+$ cation. We build a tapered variational ansatz
[\[3\]](https://pennylane.ai/qml/demos/tutorial_givens_rotations) that
prepares an entangled state by evolving the tapered Hartree-Fock state
using the tapered particle-conserving gates, i.e., the
[SingleExcitation](https://docs.pennylane.ai/en/stable/code/api/pennylane.SingleExcitation.html#)
and
[DoubleExcitation](https://docs.pennylane.ai/en/stable/code/api/pennylane.DoubleExcitation.html#)
operations tapered using
[taper_operation](https://docs.pennylane.ai/en/stable/code/api/pennylane.qchem.taper_operation.html#).


In [ ]:
singles, doubles = qp.qchem.excitations(n_electrons, len(H.wires))
tapered_doubles = [
    qp.taper_operation(qp.DoubleExcitation, generators, paulixops, paulix_sector,
                        wire_order=H.wires, op_wires=double) for double in doubles
]
tapered_singles = [
    qp.taper_operation(qp.SingleExcitation, generators, paulixops, paulix_sector,
                        wire_order=H.wires, op_wires=single) for single in singles
]

dev = qp.device("lightning.qubit", wires=H_tapered.wires)

@qp.qnode(dev, interface="jax")
def tapered_circuit(params):
    qp.BasisState(state_tapered, wires=H_tapered.wires)
    for idx, tapered_op in enumerate(tapered_doubles + tapered_singles):
        tapered_op(params[idx])
    return qp.expval(H_tapered)

We define an optimizer and the initial values of the circuit parameters
and optimize the circuit parameters with respect to the ground state
energy.


In [ ]:
import optax
import catalyst

opt = optax.sgd(learning_rate=0.8) # sgd stands for StochasticGradientDescent
init_params = jnp.zeros(len(doubles) + len(singles))

def update_step(i, params, opt_state):
    """Perform a single gradient update step"""
    grads = catalyst.grad(tapered_circuit)(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return (params, opt_state)

loss_history = []

opt_state = opt.init(init_params)
params = init_params

for i in range(1, 41):
    params, opt_state = update_step(i, params, opt_state)
    energy = tapered_circuit(params)
    if not i % 5:
        print(f"n: {i}, E: {energy:.8f} Ha, Params: {params}")

The computed energy matches the FCI energy, $-2.862595242378$ Ha, while
the number of qubits and the number of Hamiltonian terms are
significantly reduced with respect to their original values.

# Conclusions

Molecular Hamiltonians possess symmetries that can be leveraged to
reduce the number of qubits required in quantum computing simulations.
This tutorial introduces a PennyLane functionality that can be used for
qubit tapering based on $\mathbb{Z}_2$ symmetries. The procedure
includes obtaining tapered Hamiltonians and tapered reference states
that can be used in variational quantum algorithms such as VQE.
